# Notebook 01: Data Loading & Text Preprocessing

This notebook handles:
1. Loading the IMDB dataset
2. Exploring dataset structure and statistics
3. Separating real labels (for later evaluation)
4. Applying text preprocessing
5. Saving cleaned reviews for next phase

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Get the absolute path to the directory where THIS script is located
# Then go to the 'src' folder relative to that
current_dir = os.path.dirname(os.path.abspath('')) # Use os.getcwd() if in a Notebook
src_path = os.path.join(current_dir, 'src')

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from preprocess import preprocess_reviews

print("Libraries imported successfully!")

Libraries imported successfully!


## Phase 1: Load Dataset

In [2]:
# Load the IMDB dataset
df = pd.read_csv('../datasets/IMDB Dataset.csv')

print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")

Dataset loaded successfully!
Dataset shape: (50000, 2)


## Phase 2: Explore Dataset Structure

In [3]:
# Display first few rows
print("First 3 reviews:")
print(df.head(3))
print("\n" + "="*80 + "\n")

First 3 reviews:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive




In [4]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())
print("\n" + "="*80 + "\n")

Missing values:
review       0
sentiment    0
dtype: int64




In [5]:
# Check data types
print("Data types:")
print(df.dtypes)
print("\n" + "="*80 + "\n")

Data types:
review       str
sentiment    str
dtype: object




In [6]:
# Check label distribution
print("Sentiment distribution:")
print(df['sentiment'].value_counts())
print(f"\nClass balance: {df['sentiment'].value_counts(normalize=True).round(3)}")
print("\n" + "="*80 + "\n")

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Class balance: sentiment
positive    0.5
negative    0.5
Name: proportion, dtype: float64




## Phase 3: Separate Reviews and Labels

In [7]:
# Separate reviews and labels
reviews = df['review'].values

# Convert sentiment to binary labels (0 = negative, 1 = positive)
real_labels = (df['sentiment'] == 'positive').astype(int).values

print(f"Number of reviews: {len(reviews)}")
print(f"Number of labels: {len(real_labels)}")
print(f"Label distribution: {np.bincount(real_labels)}")

Number of reviews: 50000
Number of labels: 50000
Label distribution: [25000 25000]


In [8]:
# Show example review before preprocessing
print("EXAMPLE ORIGINAL REVIEW:")
print("="*80)
print(reviews[0])
print(f"\nSentiment: {real_labels[0]} (1 = positive)")

EXAMPLE ORIGINAL REVIEW:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of t

## Phase 4: Preprocess Reviews

In [9]:
# Apply preprocessing to all reviews
print("Starting preprocessing... This may take a few minutes.")
cleaned_reviews = preprocess_reviews(reviews)
print(f"\nPreprocessing completed! Processed {len(cleaned_reviews)} reviews.")

Starting preprocessing... This may take a few minutes.

Preprocessing completed! Processed 50000 reviews.


In [10]:
# Show example after preprocessing
print("SAME REVIEW AFTER PREPROCESSING:")
print("="*80)
print(cleaned_reviews[0])
print(f"\nOriginal length: {len(reviews[0].split())} words")
print(f"Cleaned length: {len(cleaned_reviews[0].split())} words")

SAME REVIEW AFTER PREPROCESSING:
reviewer mentioned watching oz episode youll hooked right exactly happened first thing struck oz brutality unflinching scene violence set right word go trust not faint hearted timid pull no punch regard drug sex violence hardcore classic use word called oz nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inwards privacy not high agenda em city home manyaryans muslim gangsta latino christian italian irish moreso scuffle death stare dodgy dealing shady agreement never far away say main appeal due fact go show wouldnt dare forget pretty picture painted mainstream audience forget charm forget romanceoz doesnt mess around first episode ever saw struck nasty surreal couldnt say ready watched developed taste oz got accustomed high level graphic violence not violence injustice crooked guard wholl sold nickel inmate wholl kill order away well mannered middle class inmate turned p

In [11]:
# Calculate preprocessing statistics
original_lengths = [len(r.split()) for r in reviews]
cleaned_lengths = [len(r.split()) for r in cleaned_reviews]

print("\nPREPROCESSING STATISTICS:")
print("="*80)
print(f"Average original review length: {np.mean(original_lengths):.2f} words")
print(f"Average cleaned review length: {np.mean(cleaned_lengths):.2f} words")
print(f"Reduction: {(1 - np.mean(cleaned_lengths)/np.mean(original_lengths))*100:.1f}%")
print(f"\nMin cleaned length: {np.min(cleaned_lengths)} words")
print(f"Max cleaned length: {np.max(cleaned_lengths)} words")
print(f"Median cleaned length: {np.median(cleaned_lengths):.0f} words")


PREPROCESSING STATISTICS:
Average original review length: 231.16 words
Average cleaned review length: 112.08 words
Reduction: 51.5%

Min cleaned length: 2 words
Max cleaned length: 1384 words
Median cleaned length: 83 words


## Phase 5: Save Preprocessed Reviews

In [12]:
# Create output directory if not exists
os.makedirs('./outputs', exist_ok=True)

# Save cleaned reviews
cleaned_df = pd.DataFrame({
    'cleaned_review': cleaned_reviews,
    'label': real_labels
})

cleaned_df.to_csv('./outputs/cleaned_reviews.csv', index=False)
print("✓ Saved cleaned_reviews.csv")

# Also save just the reviews for vectorization
pd.Series(cleaned_reviews).to_csv('./outputs/cleaned_reviews_only.csv', index=False, header=False)
np.save('./outputs/real_labels.npy', real_labels)
print("✓ Saved cleaned_reviews_only.csv")
print("✓ Saved real_labels.npy")

✓ Saved cleaned_reviews.csv
✓ Saved cleaned_reviews_only.csv
✓ Saved real_labels.npy


In [13]:
print("\n" + "="*80)
print("PREPROCESSING PHASE COMPLETE!")
print("="*80)
print(f"Total reviews processed: {len(cleaned_reviews)}")
print(f"Positive reviews: {np.sum(real_labels)}")
print(f"Negative reviews: {len(real_labels) - np.sum(real_labels)}")
print("\nNext step: Run Notebook 02 (PCA & K-Means)")


PREPROCESSING PHASE COMPLETE!
Total reviews processed: 50000
Positive reviews: 25000
Negative reviews: 25000

Next step: Run Notebook 02 (PCA & K-Means)
